# Smart Personal Assistant Using Multi-Agent System
### Assessment Project | LangGraph + LangChain + Real-Time APIs

---

**Architecture:**
```
User Query
    └── Router Agent (detects intent)
            ├── Weather Agent  (Open-Meteo API)
            ├── Crypto Agent   (CoinGecko API)
            ├── Currency Agent (ExchangeRate API)
            ├── Joke Agent     (JokeAPI)
            └── Quote Agent    (Quotable API)
                    └── Combiner Agent
                            └── Validation + Retry
                                    └── Report Generator (.txt + .json)
```

---

**Before running:** Add your `OPENAI_API_KEY` to Colab Secrets (Key icon on left panel)

## Section 1: Installation

In [ ]:
# ============================================================
# SECTION 1: INSTALLATION
# ============================================================

!pip install -qU langgraph langchain langchain-openai requests networkx matplotlib

print("All packages installed successfully!")

## Section 2: Imports

In [ ]:
# ============================================================
# SECTION 2: IMPORTS
# ============================================================

from google.colab import userdata

from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

import requests
import json
import time
import random
import datetime

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

print("All imports successful!")

## Section 3: API Key & LLM Setup

In [ ]:
# ============================================================
# SECTION 3: API KEY FROM COLAB SECRETS + LLM SETUP
# ============================================================
#
# HOW TO ADD YOUR KEY:
# 1. Click the KEY icon on the left sidebar in Colab
# 2. Add a new secret: Name = OPENAI_API_KEY, Value = sk-...
# 3. Toggle access ON for this notebook
# ============================================================

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

print("LLM (GPT-4o-mini) initialized successfully!")

## Section 4: Real-Time API Tool Functions (with Retry)

In [ ]:
# ============================================================
# SECTION 4: REAL-TIME API TOOL FUNCTIONS WITH RETRY LOGIC
# ============================================================
#
# APIs Used (all free, no key required):
#   1. Open-Meteo   -> Weather data
#   2. CoinGecko    -> Crypto prices
#   3. ExchangeRate -> Currency rates
#   4. JokeAPI      -> Random jokes
#   5. Quotable.io  -> Motivational quotes
# ============================================================

MAX_API_RETRIES = 3


def fetch_with_retry(url, headers=None, max_retries=MAX_API_RETRIES):
    """Fetch URL with exponential backoff retry."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f"    Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                time.sleep(2 ** attempt)
    return None


# ---- 1. Weather ----
def get_weather_data():
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=13.08&longitude=80.27&current_weather=true"
    )
    data = fetch_with_retry(url)
    if data and "current_weather" in data:
        w = data["current_weather"]
        code_map = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 48: "Icy fog", 51: "Light drizzle", 61: "Slight rain",
            63: "Moderate rain", 65: "Heavy rain", 80: "Slight showers",
            95: "Thunderstorm"
        }
        condition = code_map.get(w["weathercode"], f"Code {w['weathercode']}")
        return (
            f"Location     : Chennai, India (lat=13.08, lon=80.27)\n"
            f"Temperature  : {w['temperature']}°C\n"
            f"Wind Speed   : {w['windspeed']} km/h\n"
            f"Condition    : {condition}\n"
            f"Recorded At  : {w['time']}"
        )
    return "Weather data unavailable at this time."


# ---- 2. Crypto ----
def get_crypto_data():
    url = (
        "https://api.coingecko.com/api/v3/simple/price"
        "?ids=bitcoin,ethereum,dogecoin&vs_currencies=usd,inr"
    )
    data = fetch_with_retry(url)
    if data:
        lines = []
        for coin, prices in data.items():
            usd = prices.get("usd", "N/A")
            inr = prices.get("inr", "N/A")
            lines.append(f"{coin.capitalize():12s}: ${usd:>12,} USD | ₹{inr:>15,} INR")
        return "\n".join(lines)
    return "Crypto price data unavailable at this time."


# ---- 3. Currency Exchange ----
def get_currency_data():
    url = "https://open.er-api.com/v6/latest/USD"
    data = fetch_with_retry(url)
    if data and "rates" in data:
        rates = data["rates"]
        key_currencies = ["INR", "EUR", "GBP", "JPY", "AED", "SGD", "CAD", "AUD"]
        lines = ["Base Currency: 1 USD"]
        for c in key_currencies:
            if c in rates:
                lines.append(f"  1 USD = {rates[c]:>10.4f} {c}")
        lines.append(f"  Last Updated: {data.get('time_last_update_utc', 'N/A')}")
        return "\n".join(lines)
    return "Exchange rate data unavailable at this time."


# ---- 4. Joke ----
def get_joke_data():
    url = "https://v2.jokeapi.dev/joke/Any?safe-mode"
    data = fetch_with_retry(url)
    if data:
        if data.get("type") == "single":
            return data.get("joke", "")
        elif data.get("type") == "twopart":
            return f"{data.get('setup', '')}\n>> {data.get('delivery', '')}"
    # Fallback joke
    fallbacks = [
        "Why do programmers prefer dark mode?\n>> Because light attracts bugs!",
        "Why was Python angry?\n>> Because of indentation issues!",
        "AI agents never sleep.\n>> They just keep prompting!"
    ]
    return random.choice(fallbacks)


# ---- 5. Quote ----
def get_quote_data():
    url = "https://api.quotable.io/random"
    data = fetch_with_retry(url)
    if data and "content" in data:
        return f'"{data["content"]}"\n  — {data.get("author", "Unknown")}'
    # Fallback quotes
    fallbacks = [
        '"The best way to predict the future is to create it."\n  — Peter Drucker',
        '"It always seems impossible until it\'s done."\n  — Nelson Mandela',
        '"In the middle of every difficulty lies opportunity."\n  — Albert Einstein',
        '"The only way to do great work is to love what you do."\n  — Steve Jobs',
    ]
    return random.choice(fallbacks)


print("All API tool functions defined with retry logic!")

## Section 5: Shared State Definition

In [ ]:
# ============================================================
# SECTION 5: SHARED STATE DEFINITION
# ============================================================

class AssistantState(TypedDict):
    query: str
    active_agents: List[str]

    # Individual agent outputs
    weather_output: str
    crypto_output: str
    currency_output: str
    joke_output: str
    quote_output: str

    # Final combined response
    final_response: str

    # Retry tracking
    retry_count: int
    validation_passed: bool


print("AssistantState schema defined!")
print("Fields:", list(AssistantState.__annotations__.keys()))

## Section 6: Router Agent

In [ ]:
# ============================================================
# SECTION 6: ROUTER AGENT
# Detects user intent and decides which agents to activate.
# Supports multiple agents for a single query.
# ============================================================

def router_agent(state: AssistantState):
    query = state["query"].lower()
    active = []

    # Intent detection keywords
    weather_kw   = ["weather", "rain", "temperature", "sunny", "climate", "forecast", "hot", "cold", "humid"]
    crypto_kw    = ["bitcoin", "ethereum", "dogecoin", "crypto", "btc", "eth", "doge", "coin", "blockchain"]
    currency_kw  = ["currency", "exchange", "usd", "inr", "eur", "convert", "rate", "dollar", "rupee", "forex"]
    joke_kw      = ["joke", "funny", "laugh", "humor", "comedy", "entertain"]
    quote_kw     = ["quote", "motivat", "inspir", "wisdom", "thought", "encourage"]

    if any(kw in query for kw in weather_kw):
        active.append("weather")

    if any(kw in query for kw in crypto_kw):
        active.append("crypto")

    if any(kw in query for kw in currency_kw):
        active.append("currency")

    if any(kw in query for kw in joke_kw):
        active.append("joke")

    if any(kw in query for kw in quote_kw):
        active.append("quote")

    # Default: activate all agents if no specific intent detected
    if not active:
        active = ["weather", "crypto", "currency", "joke", "quote"]
        print("  No specific intent detected. Activating ALL agents.")

    print(f"\n[Router Agent] Detected Active Agents: {active}")

    return {
        "active_agents": active,
        "weather_output": "",
        "crypto_output": "",
        "currency_output": "",
        "joke_output": "",
        "quote_output": "",
        "retry_count": 0,
        "validation_passed": False
    }


print("Router Agent defined!")

## Section 7: Specialized Agents (5 Agents)

In [ ]:
# ============================================================
# SECTION 7: SPECIALIZED AGENTS
# Each agent checks if it is active before calling the API.
# All API calls include retry logic.
# ============================================================


# ---- Agent 1: Weather Agent ----
def weather_agent(state: AssistantState):
    if "weather" not in state["active_agents"]:
        return {"weather_output": ""}

    print("  [Weather Agent]   Fetching weather from Open-Meteo API...")
    result = get_weather_data()
    print(f"  [Weather Agent]   Done.")
    return {"weather_output": result}


# ---- Agent 2: Crypto Agent ----
def crypto_agent(state: AssistantState):
    if "crypto" not in state["active_agents"]:
        return {"crypto_output": ""}

    print("  [Crypto Agent]    Fetching prices from CoinGecko API...")
    result = get_crypto_data()
    print(f"  [Crypto Agent]    Done.")
    return {"crypto_output": result}


# ---- Agent 3: Currency Agent ----
def currency_agent(state: AssistantState):
    if "currency" not in state["active_agents"]:
        return {"currency_output": ""}

    print("  [Currency Agent]  Fetching exchange rates from ExchangeRate API...")
    result = get_currency_data()
    print(f"  [Currency Agent]  Done.")
    return {"currency_output": result}


# ---- Agent 4: Joke Agent ----
def joke_agent(state: AssistantState):
    if "joke" not in state["active_agents"]:
        return {"joke_output": ""}

    print("  [Joke Agent]      Fetching a joke from JokeAPI...")
    result = get_joke_data()
    print(f"  [Joke Agent]      Done.")
    return {"joke_output": result}


# ---- Agent 5: Quote Agent ----
def quote_agent(state: AssistantState):
    if "quote" not in state["active_agents"]:
        return {"quote_output": ""}

    print("  [Quote Agent]     Fetching a quote from Quotable API...")
    result = get_quote_data()
    print(f"  [Quote Agent]     Done.")
    return {"quote_output": result}


print("All 5 Specialized Agents defined!")

## Section 8: Combiner Agent

In [ ]:
# ============================================================
# SECTION 8: COMBINER AGENT
# Collects outputs from all active agents and uses LLM
# to generate a single clean, coherent final response.
# ============================================================

def combiner_agent(state: AssistantState):
    print("\n[Combiner Agent] Merging all agent outputs via LLM...")

    sections = []

    if state["weather_output"]:
        sections.append(f"=== WEATHER DATA (Chennai) ===\n{state['weather_output']}")

    if state["crypto_output"]:
        sections.append(f"=== CRYPTO PRICES ===\n{state['crypto_output']}")

    if state["currency_output"]:
        sections.append(f"=== CURRENCY EXCHANGE RATES ===\n{state['currency_output']}")

    if state["joke_output"]:
        sections.append(f"=== JOKE ===\n{state['joke_output']}")

    if state["quote_output"]:
        sections.append(f"=== MOTIVATIONAL QUOTE ===\n{state['quote_output']}")

    combined_raw = "\n\n".join(sections)

    prompt = f"""You are a Smart Personal AI Assistant.

The user asked: \"{state['query']}\"

Here is the real-time data collected from multiple specialized agents:

{combined_raw}

Generate a clean, structured, and helpful final response that:
1. Directly addresses the user's query
2. Presents all relevant information clearly with proper formatting
3. Removes any redundant information
4. Is friendly and easy to read
"""

    response = llm.invoke([HumanMessage(content=prompt)])

    print("[Combiner Agent] Final response ready.")
    return {"final_response": response.content}


print("Combiner Agent defined!")

## Section 9: Validation Agent & Retry Logic

In [ ]:
# ============================================================
# SECTION 9: VALIDATION AGENT + RETRY LOGIC
# Validates the final response quality.
# Routes back to combiner if validation fails (up to 3 retries).
# ============================================================

MAX_RETRIES = 3


def validation_agent(state: AssistantState):
    retry = state["retry_count"] + 1
    response = state["final_response"].strip()

    # Validation: response must be non-empty and at least 50 characters
    is_valid = len(response) > 50

    status = "PASSED" if is_valid else "FAILED"
    print(f"\n[Validation Agent] Attempt {retry}/{MAX_RETRIES} -> {status} (length: {len(response)} chars)")

    return {
        "retry_count": retry,
        "validation_passed": is_valid
    }


def retry_router(state: AssistantState) -> str:
    """Conditional edge: retry combiner or proceed to save."""
    if state["validation_passed"]:
        print("[Retry Router] Validation PASSED -> Saving report.")
        return "save"

    if state["retry_count"] >= MAX_RETRIES:
        print(f"[Retry Router] Max retries ({MAX_RETRIES}) reached -> Saving report anyway.")
        return "save"

    print(f"[Retry Router] Validation FAILED -> Retrying combiner (attempt {state['retry_count'] + 1}).")
    return "retry"


print("Validation Agent and Retry Router defined!")

## Section 10: Report Generator Agent

In [ ]:
# ============================================================
# SECTION 10: REPORT GENERATOR AGENT
# Saves a timestamped .txt report and .json output file.
# ============================================================

def report_agent(state: AssistantState):
    now = datetime.datetime.now()
    timestamp_str = now.strftime("%Y_%m_%d_%H_%M")
    timestamp_readable = now.strftime("%Y-%m-%d %H:%M:%S")

    txt_filename  = f"report_{timestamp_str}.txt"
    json_filename = f"output_{timestamp_str}.json"

    # ---- Build TXT Report ----
    sep = "=" * 72
    thin = "-" * 72

    report_lines = [
        sep,
        "       SMART PERSONAL ASSISTANT  —  EXECUTION REPORT",
        sep,
        f"  Execution Timestamp : {timestamp_readable}",
        f"  Report File         : {txt_filename}",
        sep,
        "",
        "USER QUERY",
        thin,
        state["query"],
        "",
        "SELECTED AGENTS",
        thin,
        ", ".join(state["active_agents"]),
        "",
        "RAW API OUTPUTS",
        thin,
    ]

    if state["weather_output"]:
        report_lines += ["", "[Weather Agent — Open-Meteo API]", state["weather_output"]]

    if state["crypto_output"]:
        report_lines += ["", "[Crypto Agent — CoinGecko API]", state["crypto_output"]]

    if state["currency_output"]:
        report_lines += ["", "[Currency Agent — ExchangeRate API]", state["currency_output"]]

    if state["joke_output"]:
        report_lines += ["", "[Joke Agent — JokeAPI]", state["joke_output"]]

    if state["quote_output"]:
        report_lines += ["", "[Quote Agent — Quotable API]", state["quote_output"]]

    report_lines += [
        "",
        sep,
        "",
        "FINAL COMBINED RESPONSE (LLM Generated)",
        thin,
        state["final_response"],
        "",
        sep,
        f"  Total Retries : {state['retry_count']}",
        f"  Validation    : {'PASSED' if state['validation_passed'] else 'MAX RETRIES REACHED'}",
        sep,
    ]

    report_content = "\n".join(report_lines)

    with open(txt_filename, "w", encoding="utf-8") as f:
        f.write(report_content)

    # ---- Build JSON Output ----
    json_data = {
        "execution_timestamp": timestamp_readable,
        "query": state["query"],
        "active_agents": state["active_agents"],
        "api_outputs": {
            "weather": state["weather_output"],
            "crypto": state["crypto_output"],
            "currency": state["currency_output"],
            "joke": state["joke_output"],
            "quote": state["quote_output"]
        },
        "final_response": state["final_response"],
        "retry_count": state["retry_count"],
        "validation_passed": state["validation_passed"]
    }

    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)

    print(f"\n[Report Agent] TXT  saved: {txt_filename}")
    print(f"[Report Agent] JSON saved: {json_filename}")

    return state


print("Report Generator Agent defined!")

## Section 11: Build LangGraph Workflow

In [ ]:
# ============================================================
# SECTION 11: BUILD LANGGRAPH WORKFLOW
# ============================================================
#
# Flow:
#   router_agent
#       -> weather_agent -> crypto_agent -> currency_agent
#       -> joke_agent -> quote_agent
#       -> combiner_agent
#       -> validation_agent
#             |-- retry --> combiner_agent (loop)
#             |-- save  --> report_agent
#                               --> END
# ============================================================

builder = StateGraph(AssistantState)

# Register all nodes
builder.add_node("router_agent",     router_agent)
builder.add_node("weather_agent",    weather_agent)
builder.add_node("crypto_agent",     crypto_agent)
builder.add_node("currency_agent",   currency_agent)
builder.add_node("joke_agent",       joke_agent)
builder.add_node("quote_agent",      quote_agent)
builder.add_node("combiner_agent",   combiner_agent)
builder.add_node("validation_agent", validation_agent)
builder.add_node("report_agent",     report_agent)

# Entry point
builder.set_entry_point("router_agent")

# Sequential edges through specialized agents
builder.add_edge("router_agent",   "weather_agent")
builder.add_edge("weather_agent",  "crypto_agent")
builder.add_edge("crypto_agent",   "currency_agent")
builder.add_edge("currency_agent", "joke_agent")
builder.add_edge("joke_agent",     "quote_agent")
builder.add_edge("quote_agent",    "combiner_agent")
builder.add_edge("combiner_agent", "validation_agent")

# Conditional edge: retry or save
builder.add_conditional_edges(
    "validation_agent",
    retry_router,
    {
        "retry": "combiner_agent",
        "save":  "report_agent"
    }
)

builder.add_edge("report_agent", END)

# Compile the graph
graph = builder.compile()

print("LangGraph workflow compiled successfully!")
print("\nNodes registered:")
print("  router_agent -> weather_agent -> crypto_agent -> currency_agent")
print("  -> joke_agent -> quote_agent -> combiner_agent")
print("  -> validation_agent --[retry]--> combiner_agent")
print("                      --[save] --> report_agent -> END")

## Section 12: Workflow Visualization

In [ ]:
# ============================================================
# SECTION 12: WORKFLOW VISUALIZATION
# ============================================================

G = nx.DiGraph()

edges = [
    ("Router\nAgent",      "Weather\nAgent"),
    ("Weather\nAgent",     "Crypto\nAgent"),
    ("Crypto\nAgent",      "Currency\nAgent"),
    ("Currency\nAgent",    "Joke\nAgent"),
    ("Joke\nAgent",        "Quote\nAgent"),
    ("Quote\nAgent",       "Combiner\nAgent"),
    ("Combiner\nAgent",    "Validation\nAgent"),
    ("Validation\nAgent",  "Combiner\nAgent"),   # retry loop
    ("Validation\nAgent",  "Report\nAgent"),
    ("Report\nAgent",      "END"),
]

G.add_edges_from(edges)

# Custom positions for clear layout
pos = {
    "Router\nAgent":     (5.0, 9.0),
    "Weather\nAgent":    (1.0, 7.0),
    "Crypto\nAgent":     (3.0, 7.0),
    "Currency\nAgent":   (5.0, 7.0),
    "Joke\nAgent":       (7.0, 7.0),
    "Quote\nAgent":      (9.0, 7.0),
    "Combiner\nAgent":   (5.0, 5.0),
    "Validation\nAgent": (5.0, 3.0),
    "Report\nAgent":     (5.0, 1.0),
    "END":               (5.0, -0.5),
}

node_colors = {
    "Router\nAgent":     "#FF6B6B",
    "Weather\nAgent":    "#4ECDC4",
    "Crypto\nAgent":     "#45B7D1",
    "Currency\nAgent":   "#96CEB4",
    "Joke\nAgent":       "#FFD93D",
    "Quote\nAgent":      "#C3A6FF",
    "Combiner\nAgent":   "#FFA07A",
    "Validation\nAgent": "#98D8C8",
    "Report\nAgent":     "#F4A261",
    "END":               "#A9A9A9",
}

colors = [node_colors[n] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))

nx.draw(
    G, pos,
    ax=ax,
    with_labels=True,
    node_size=3500,
    node_color=colors,
    font_size=9,
    font_weight="bold",
    arrows=True,
    arrowsize=25,
    edge_color="#444444",
    width=2,
    connectionstyle="arc3,rad=0.1"
)

# Legend
legend_items = [
    mpatches.Patch(color="#FF6B6B", label="Router Agent"),
    mpatches.Patch(color="#4ECDC4", label="Specialized Agents (5)"),
    mpatches.Patch(color="#FFA07A", label="Combiner Agent"),
    mpatches.Patch(color="#98D8C8", label="Validation + Retry"),
    mpatches.Patch(color="#F4A261", label="Report Generator"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=10)

ax.set_title("Smart Personal Assistant — Multi-Agent Workflow", fontsize=15, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()
print("Workflow visualization complete!")

## Section 13: Run the System

In [ ]:
# ============================================================
# SECTION 13: RUN THE SMART PERSONAL ASSISTANT
# ============================================================
#
# Modify the query below to test different scenarios:
#
#   Weather only : "What is the weather today?"
#   Crypto only  : "What is Bitcoin price right now?"
#   Currency only: "Show me USD to INR exchange rate"
#   Joke only    : "Tell me a funny joke"
#   Quote only   : "Give me a motivational quote"
#   Mixed query  : (see below — tests multi-agent routing)
# ============================================================

query = """
What is the weather in Chennai today?
Also show me Bitcoin and Ethereum prices.
And give me a motivational quote to start my day.
"""

print("=" * 60)
print("   SMART PERSONAL ASSISTANT")
print("=" * 60)
print(f"Query: {query.strip()}")
print("=" * 60)
print("\nStarting multi-agent workflow...\n")

result = graph.invoke({
    "query": query.strip(),
    "active_agents": [],
    "weather_output": "",
    "crypto_output": "",
    "currency_output": "",
    "joke_output": "",
    "quote_output": "",
    "final_response": "",
    "retry_count": 0,
    "validation_passed": False
})

print("\n" + "=" * 60)
print("   FINAL RESPONSE")
print("=" * 60)
print(result["final_response"])
print("\n" + "=" * 60)
print(f"Active Agents : {result['active_agents']}")
print(f"Total Retries : {result['retry_count']}")
print(f"Validation    : {'PASSED' if result['validation_passed'] else 'MAX RETRIES'}")
print("=" * 60)

## Section 14: Test All Agents (All-in-One Query)

In [ ]:
# ============================================================
# SECTION 14: TEST ALL AGENTS WITH A COMPREHENSIVE QUERY
# This tests the full system with all 5 agents active.
# ============================================================

full_query = """
Give me today's weather forecast, current Bitcoin and Ethereum crypto prices,
USD to INR exchange rate, a funny joke, and an inspirational quote.
"""

print("=" * 60)
print("   FULL SYSTEM TEST — ALL 5 AGENTS")
print("=" * 60)
print(f"Query: {full_query.strip()}")
print("=" * 60 + "\n")

full_result = graph.invoke({
    "query": full_query.strip(),
    "active_agents": [],
    "weather_output": "",
    "crypto_output": "",
    "currency_output": "",
    "joke_output": "",
    "quote_output": "",
    "final_response": "",
    "retry_count": 0,
    "validation_passed": False
})

print("\n" + "=" * 60)
print("   FINAL COMBINED RESPONSE")
print("=" * 60)
print(full_result["final_response"])

print("\n" + "=" * 60)
print("   RAW API OUTPUTS SUMMARY")
print("=" * 60)

if full_result["weather_output"]:
    print("\n[Weather]")
    print(full_result["weather_output"])

if full_result["crypto_output"]:
    print("\n[Crypto]")
    print(full_result["crypto_output"])

if full_result["currency_output"]:
    print("\n[Currency]")
    print(full_result["currency_output"])

if full_result["joke_output"]:
    print("\n[Joke]")
    print(full_result["joke_output"])

if full_result["quote_output"]:
    print("\n[Quote]")
    print(full_result["quote_output"])

print("\n" + "=" * 60)
print("Report files saved in Colab file system.")
print("Check the folder icon on the left sidebar.")
print("=" * 60)